## ?????? ?????????

????????? ??? ??????????? ?????????? ??? ????????? ??????, ????????????, ?????????? ??????? ? ?????? ????????. ???????? ?????????? `pandas`, `numpy`, `matplotlib`, ? ????? ?????? ?? `scikit-learn` ? `xgboost`.


In [ ]:
from xgboost import XGBClassifier, XGBRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.svm import SVC, SVR
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import math
import itertools
from pprint import pprint
import joblib

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import KFold, cross_validate, GridSearchCV, cross_val_score, RandomizedSearchCV 
from sklearn.model_selection import cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.metrics import make_scorer
from sklearn.metrics import mean_absolute_error , mean_squared_error, r2_score
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.base import BaseEstimator
from sklearn.base import ClassifierMixin
from sklearn.base import TransformerMixin
from sklearn.base import clone
from sklearn.model_selection._split import check_cv
from utils import *

## ???????? ??????? ?????

????????? ????? ? ???????? ??????????? ??? ????????? ? ???????? ???????. ??????????????? ???????? `CASRN`.


In [ ]:
train_labels = pd.read_csv('../data/processed/train_labels.csv', index_col = 'CASRN')

train_ecfp6_bits = pd.read_csv('../data/Bmodel_features/modeling_train_ecfp6_bits.csv', index_col='CASRN')
train_ecfp6_counts = pd.read_csv('../data/Bmodel_features/modeling_train_ecfp6_counts.csv', index_col='CASRN')
train_maccs = pd.read_csv('../data/Bmodel_features/modeling_train_maccs.csv', index_col='CASRN')
train_rdkit2d = pd.read_csv('../data/Bmodel_features/modeling_train_rdkit2d.csv', index_col='CASRN')
train_mordred = pd.read_csv('../data/Bmodel_features/modeling_train_mordred.csv', index_col='CASRN')

??????? ?????????? ???????

In [ ]:
knn_reg = KNeighborsRegressor()
svr = SVR()
xgb_reg = XGBRegressor(random_state =123, n_jobs=6 ,objective ='reg:squarederror')
rf_reg =  RandomForestRegressor(random_state =123, n_jobs=6)

svc = SVC(random_state =42)
knn_clf = KNeighborsClassifier()
xgb_clf = XGBClassifier(random_state =123, n_jobs=6)
rf_clf =  RandomForestClassifier(random_state =123, n_jobs=6)

?????? ??????? ?? ??????????????? ??? ???????

In [ ]:
knn_grid_parameters_des = {'n_neighbors': [5,9,15,19,25,35,45,55,71], 'weights': ['distance'],'p': [1,2]}
knn_grid_parameters_fp = {'n_neighbors': [5,9,15,19,23,25,35,45,55,71], 'weights': ['distance'],
                          'metric': ['jaccard', 'dice', 'rogerstanimoto']}
knn_grid_parameters_fpcounts = {'n_neighbors': [5,9,15,19,25,35,45,55,71], 'weights': ['distance'],
                          'metric': ['hamming', 'canberra', 'braycurtis']}


In [ ]:
svm_grid_parameters = [
  {'C': [0.01, 0.1, 1, 10, 100, 200, 400, 1000], 'kernel': ['linear']},
  {'C': [0.01, 0.1, 1, 10, 100, 200, 400, 1000], 'gamma': [100,10,1,1e-1,1e-2, 1e-3], 'kernel': ['rbf']},
 ]


In [ ]:
svm_grid_parameters = { 'gamma': [10,1,1e-1,1e-2],
                     'C': [1, 10, 100]}

In [ ]:
svm_grid_parameters_1 = { 'gamma': [1,1e-1],
                     'C': [1, 10]}


In [ ]:
svm_grid_parameters_2 = [
  {'C': [1, 10], 'kernel': ['linear']},
  {'C': [1, 10], 'gamma': [1,1e-1], 'kernel': ['rbf']},
 ]

In [ ]:
n_estimators = [int(x) for x in np.linspace(start = 500, stop = 1500, num = 2)]
max_features = ['log2', 'sqrt']
max_depth = [int(x) for x in np.linspace(5, 80, num = 6)]
max_depth.append(None)
min_samples_split = [2, 5, 10]
min_samples_leaf = [2, 4, 6]
bootstrap = [True, False]

rf_grid_parameters = {'n_estimators': n_estimators,
               'max_features': max_features,
               'max_depth': max_depth,
               'min_samples_split': min_samples_split,
               'min_samples_leaf': min_samples_leaf,
               'bootstrap': bootstrap}

pprint(rf_grid_parameters)

In [ ]:
xgb_grid_parameters = {
    'learning_rate': [0.01,0.1],
    'max_depth': [3,6,10],
    'min_child_weight':[1,3,5], 
    'gamma':[0,1,5],  
    'subsample':[i/10.0 for i in range(6,11)],
    'colsample_bytree':[i/10.0 for i in range(5,11)], 
    'n_estimators': [500,1500]}

??????? ??????? ??? ?????????????????? ?????? ?????????? ???????????

In [ ]:
def result_model_selection(results, name):
    df_results = pd.DataFrame({'model': [name] * len(results.cv_results_['params']),
                               'params': results.cv_results_['params'],
                              'mean score': results.cv_results_['mean_test_score'],
                              'std score': results.cv_results_['std_test_score'],
                               'rank': results.cv_results_['rank_test_score']
                              })
    return df_results

## ???????? ??????

kNN
Endpoint 1: VeryToxic

In [ ]:
encoder_verytoxic = joblib.load('../encoder_models/encoder_verytoxic.joblib')


In [ ]:
%%time
a,b,c,d,e = prepare_input(train_labels, train_ecfp6_bits, target = 'verytoxic', encoder = encoder_verytoxic)

vt_knn_ecfp6bits = model_selection(knn_clf, knn_grid_parameters_fp, a, c, scoring = 'roc_auc', cv=5, GridSearch = True)


In [ ]:
%%time
a,b,c,d,e = prepare_input(train_labels, train_maccs, target = 'verytoxic', encoder = encoder_verytoxic)

vt_knn_maccs = model_selection(knn_clf, knn_grid_parameters_fp, a, c, scoring = 'roc_auc', cv=5, GridSearch = True)

In [ ]:
%%time
a,b,c,d,e = prepare_input(train_labels, train_ecfp6_counts, target = 'verytoxic', encoder = encoder_verytoxic)

vt_knn_ecfp6count = model_selection(knn_clf, knn_grid_parameters_fpcounts, a, c, scoring = 'roc_auc', cv=5, GridSearch = True)


In [ ]:
%%time
a,b,c,d,e = prepare_input(train_labels, train_rdkit2d, target = 'verytoxic', encoder = encoder_verytoxic)

vt_knn_rdkit2d = model_selection(knn_clf, knn_grid_parameters_des, a, c, scoring = 'roc_auc', cv=5, GridSearch = True)


In [ ]:
%%time
a,b,c,d,e = prepare_input(train_labels, train_mordred, target = 'verytoxic', encoder = encoder_verytoxic)

vt_knn_mordred = model_selection(knn_clf, knn_grid_parameters_des, a, c, scoring = 'roc_auc', cv=5, GridSearch = True)

Endpoint 2: toxic

In [ ]:
encoder_toxic = joblib.load('../encoder_models/encoder_toxic.joblib')

In [ ]:
%%time
a,b,c,d,e = prepare_input(train_labels, train_ecfp6_bits, target = 'toxic', encoder = encoder_toxic)

t_knn_ecfp6bits = model_selection(knn_clf, knn_grid_parameters_fp, a, c, scoring = 'roc_auc', cv=5, GridSearch = True)

In [ ]:
%%time
a,b,c,d,e = prepare_input(train_labels, train_maccs, target = 'toxic', encoder = encoder_toxic)

t_knn_maccs = model_selection(knn_clf, knn_grid_parameters_fp, a, c, scoring = 'roc_auc', cv=5, GridSearch = True)

In [ ]:
%%time
a,b,c,d,e = prepare_input(train_labels, train_ecfp6_counts, target = 'toxic', encoder = encoder_toxic)

t_knn_ecfp6count = model_selection(knn_clf, knn_grid_parameters_fpcounts, a, c, scoring = 'roc_auc', cv=5, GridSearch = True)


In [ ]:
%%time
a,b,c,d,e = prepare_input(train_labels, train_rdkit2d, target = 'toxic', encoder = encoder_toxic)

t_knn_rdkit2d = model_selection(knn_clf, knn_grid_parameters_des, a, c, scoring = 'roc_auc', cv=5, GridSearch = True)

In [ ]:
%%time
a,b,c,d,e = prepare_input(train_labels, train_mordred, target = 'toxic', encoder = encoder_toxic)

t_knn_mordred = model_selection(knn_clf, knn_grid_parameters_des, a, c, scoring = 'roc_auc', cv=5, GridSearch = True)

Endpoint 3: EPA

In [ ]:
encoder_epa = joblib.load('../encoder_models/encoder_epa.joblib')


In [ ]:
%%time
a,b,c,d,e = prepare_input(train_labels, train_ecfp6_bits, target = 'EPA_category', encoder = encoder_epa)

epa_knn_ecfp6bits = model_selection(knn_clf, knn_grid_parameters_fp, a, c, scoring = 'f1_weighted', cv=5, GridSearch = True)



In [ ]:
%%time
a,b,c,d,e = prepare_input(train_labels, train_maccs, target = 'EPA_category', encoder = encoder_epa)

epa_knn_maccs = model_selection(knn_clf, knn_grid_parameters_fp, a, c, scoring = 'f1_weighted', cv=5, GridSearch = True)


In [ ]:
%%time
a,b,c,d,e = prepare_input(train_labels, train_ecfp6_counts, target = 'EPA_category', encoder = encoder_epa)

epa_knn_ecfp6count = model_selection(knn_clf, knn_grid_parameters_fpcounts, a, c, scoring = 'f1_weighted', cv=5, GridSearch = True)


In [ ]:
%%time
a,b,c,d,e = prepare_input(train_labels, train_rdkit2d, target = 'EPA_category', encoder = encoder_epa)

epa_knn_rdkit2d = model_selection(knn_clf, knn_grid_parameters_des, a, c, scoring = 'f1_weighted', cv=5, GridSearch = True)


In [ ]:
%%time
a,b,c,d,e = prepare_input(train_labels, train_mordred, target = 'EPA_category', encoder = encoder_epa)

epa_knn_mordred = model_selection(knn_clf, knn_grid_parameters_des, a, c, scoring = 'f1_weighted', cv=5, GridSearch = True)


Endpoint 4: logLD50

In [ ]:
%%time
a,b,c,d,e = prepare_input(train_labels, train_ecfp6_bits, target = 'logLD50_mmolkg', encoder = None)

ld50_knn_ecfp6bits = model_selection(knn_reg, knn_grid_parameters_fp, a, c, scoring = 'neg_mean_squared_error', cv=5, GridSearch = True)


In [ ]:
%%time
a,b,c,d,e = prepare_input(train_labels, train_maccs, target = 'logLD50_mmolkg', encoder = None)

ld50_knn_maccs = model_selection(knn_reg, knn_grid_parameters_fp, a, c, scoring = 'neg_mean_squared_error', cv=5, GridSearch = True)



In [ ]:
%%time
a,b,c,d,e = prepare_input(train_labels, train_ecfp6_counts, target = 'logLD50_mmolkg', encoder = None)

ld50_knn_ecfp6count = model_selection(knn_reg, knn_grid_parameters_fpcounts, a, c, scoring = 'neg_mean_squared_error', cv=5, GridSearch = True)


In [ ]:
%%time
a,b,c,d,e = prepare_input(train_labels, train_rdkit2d, target = 'logLD50_mmolkg', encoder = None)

ld50_knn_rdkit2d = model_selection(knn_reg, knn_grid_parameters_des, a, c, scoring = 'neg_mean_squared_error', cv=5, GridSearch = True)


In [ ]:
%%time
a,b,c,d,e = prepare_input(train_labels, train_mordred, target = 'logLD50_mmolkg', encoder = None)

ld50_knn_mordred = model_selection(knn_reg, knn_grid_parameters_des, a, c, scoring = 'neg_mean_squared_error', cv=5, GridSearch = True)


SVM
Endpoint 1: verytoxic
Endpoint 1: toxic

In [ ]:
encoder_toxic = joblib.load('../encoder_models/encoder_toxic.joblib')

In [ ]:
%%time
a,b,c,d,e = prepare_input(train_labels, train_ecfp6_bits, target = 'toxic', encoder = encoder_toxic)

t_svc_ecfp6bits = model_selection(svc, svm_grid_parameters, a, c, scoring = 'roc_auc', cv=5, GridSearch = True, n_jobs=6)


In [ ]:
%%time
a,b,c,d,e = prepare_input(train_labels, train_maccs, target = 'toxic', encoder = encoder_toxic)

t_svc_maccs = model_selection(svc, svm_grid_parameters, a, c, scoring = 'roc_auc', cv=5, GridSearch = True, n_jobs=1)


In [ ]:
%%time
a,b,c,d,e = prepare_input(train_labels, train_ecfp6_counts, target = 'toxic', encoder = encoder_toxic)

t_svc_ecfp6counts = model_selection(svc, svm_grid_parameters, a, c, scoring = 'roc_auc', cv=5, GridSearch = True, n_jobs=6)


In [ ]:

%%time
a,b,c,d,e = prepare_input(train_labels, train_rdkit2d, target = 'toxic', encoder = encoder_toxic)

t_svc_rdkit2d = model_selection(svc, svm_grid_parameters, a, c, scoring = 'roc_auc', cv=5, GridSearch = True, n_jobs=1)


In [ ]:
%%time
a,b,c,d,e = prepare_input(train_labels, train_mordred, target = 'toxic', encoder = encoder_toxic)

t_svc_mordred = model_selection(svc, svm_grid_parameters, a, c, scoring = 'roc_auc', cv=5, GridSearch = True, n_jobs=1)


Endpoint 2: EPA

In [ ]:
encoder_epa = joblib.load('../encoder_models/encoder_epa.joblib')

In [ ]:
%%time
a,b,c,d,e = prepare_input(train_labels, train_ecfp6_bits, target = 'EPA_category', encoder = encoder_epa)

epa_svc_ecfp6bits = model_selection(svc, svm_grid_parameters, a, c, scoring = 'f1_weighted', cv=5, GridSearch = True, n_jobs=6)


In [ ]:
%%time
a,b,c,d,e = prepare_input(train_labels, train_ecfp6_counts, target = 'EPA_category', encoder = encoder_epa)

epa_svc_ecfp6counts = model_selection(svc, svm_grid_parameters, a, c, scoring = 'f1_weighted', cv=5, GridSearch = True, n_jobs=6)


In [ ]:
%%time
a,b,c,d,e = prepare_input(train_labels, train_maccs, target = 'EPA_category', encoder = encoder_epa)

epa_svc_maccs = model_selection(svc, svm_grid_parameters, a, c, scoring = 'f1_weighted', cv=5, GridSearch = True)


In [ ]:
%%time
a,b,c,d,e = prepare_input(train_labels, train_rdkit2d, target = 'EPA_category', encoder = encoder_epa)

epa_svc_rdkit2d = model_selection(svc, svm_grid_parameters, a, c, scoring = 'f1_weighted', cv=5, GridSearch = True)



In [ ]:
%%time
a,b,c,d,e = prepare_input(train_labels, train_mordred, target = 'EPA_category', encoder = encoder_epa)

epa_svc_mordred = model_selection(svc, svm_grid_parameters, a, c, scoring = 'f1_weighted', cv=5, GridSearch = True)


In [ ]:
df_epa_svc_mordred = result_model_selection(epa_svc_mordred, 'epa_svc_mordred')

Endpoint 3: logLD50

In [ ]:
%%time
a,b,c,d,e = prepare_input(train_labels, train_ecfp6_bits, target = 'logLD50_mmolkg', encoder = None)

ld50_svr_ecfp6bits = model_selection(svr, svm_grid_parameters, a, c, scoring = 'neg_mean_squared_error', cv=5, GridSearch = True, n_jobs = 6)


In [ ]:
%%time
a,b,c,d,e = prepare_input(train_labels, train_ecfp6_counts, target = 'logLD50_mmolkg', encoder = None)

ld50_svr_ecfp6counts = model_selection(svr, svm_grid_parameters, a, c, scoring = 'neg_mean_squared_error', cv=5, GridSearch = True, n_jobs = 6)



In [ ]:
%%time
a,b,c,d,e = prepare_input(train_labels, train_maccs, target = 'logLD50_mmolkg', encoder = None)

ld50_svr_maccs = model_selection(svr, svm_grid_parameters, a, c, scoring = 'neg_mean_squared_error', cv=5, GridSearch = True)


In [ ]:
%%time
a,b,c,d,e = prepare_input(train_labels, train_rdkit2d, target = 'logLD50_mmolkg', encoder = None)

ld50_svr_rdkit2d = model_selection(svr, svm_grid_parameters, a, c, scoring = 'neg_mean_squared_error', cv=5, GridSearch = True)


In [ ]:
%%time
a,b,c,d,e = prepare_input(train_labels, train_mordred, target = 'logLD50_mmolkg', encoder = None)

ld50_svr_mordred = model_selection(svr, svm_grid_parameters, a, c, scoring = 'neg_mean_squared_error', cv=5, GridSearch = True)


Random Forest Model
Endpoint 1: toxic

In [ ]:
encoder_toxic = joblib.load('../encoder_models/encoder_toxic.joblib')

In [ ]:
%%time
a,b,c,d,e = prepare_input(train_labels, train_ecfp6_bits, target = 'toxic', encoder = encoder_toxic)

t_rf_ecfp6bits = model_selection(rf_clf, rf_grid_parameters, a, c, scoring = 'roc_auc', cv=5, GridSearch = False, n_iter=20,n_jobs=1)


In [ ]:
%%time
a,b,c,d,e = prepare_input(train_labels, train_rdkit2d, target = 'toxic', encoder = encoder_toxic)

t_rf_rdkit2d = model_selection(rf_clf, rf_grid_parameters, a, c, scoring = 'roc_auc', cv=5, GridSearch = False, n_iter=30,n_jobs=1)


In [ ]:
%%time
a,b,c,d,e = prepare_input(train_labels, train_mordred, target = 'toxic', encoder = encoder_toxic)

t_rf_mordred = model_selection(rf_clf, rf_grid_parameters, a, c, scoring = 'roc_auc', cv=5, GridSearch = False, n_iter=30,n_jobs=1)


In [ ]:
%%time
a,b,c,d,e = prepare_input(train_labels, train_ecfp6_counts, target = 'toxic', encoder = encoder_toxic)

t_rf_ecfp6count = model_selection(rf_clf, rf_grid_parameters, a, c, scoring = 'roc_auc', cv=5, GridSearch = False, n_iter=20,n_jobs=1)


In [ ]:
%%time
a,b,c,d,e = prepare_input(train_labels, train_maccs, target = 'toxic', encoder = encoder_toxic)

t_rf_maccs = model_selection(rf_clf, rf_grid_parameters, a, c, scoring = 'roc_auc', cv=5, GridSearch = False, n_iter=30,n_jobs=1)

Endpoint 2: EPA

In [ ]:
encoder_epa = joblib.load('../encoder_models/encoder_epa.joblib')

In [ ]:
%%time
a,b,c,d,e = prepare_input(train_labels, train_ecfp6_bits, target = 'EPA_category', encoder = encoder_epa)

epa_rf_ecfp6bits = model_selection(rf_clf, rf_grid_parameters, a, c, scoring = 'f1_weighted', cv=5, GridSearch = False, n_iter=30,n_jobs=1)


In [ ]:
%%time
a,b,c,d,e = prepare_input(train_labels, train_maccs, target = 'EPA_category', encoder = encoder_epa)

epa_rf_maccs = model_selection(rf_clf, rf_grid_parameters, a, c, scoring = 'f1_weighted', cv=5, GridSearch = False, n_iter=30,n_jobs=1)


In [ ]:
%%time
a,b,c,d,e = prepare_input(train_labels, train_ecfp6_counts, target = 'EPA_category', encoder = encoder_epa)

epa_rf_ecfp6count = model_selection(rf_clf, rf_grid_parameters, a, c, scoring = 'f1_weighted', cv=5, GridSearch = False, n_iter=30,n_jobs=1)


In [ ]:
%%time
a,b,c,d,e = prepare_input(train_labels, train_rdkit2d, target = 'EPA_category', encoder = encoder_epa)

epa_rf_rdkit2d = model_selection(rf_clf, rf_grid_parameters, a, c, scoring = 'f1_weighted', cv=5, GridSearch = False, n_iter=30,n_jobs=1)


In [ ]:
%%time
a,b,c,d,e = prepare_input(train_labels, train_mordred, target = 'EPA_category', encoder = encoder_epa)

epa_rf_mordred = model_selection(rf_clf, rf_grid_parameters, a, c, scoring = 'f1_weighted', cv=5, GridSearch = False, n_iter=30,n_jobs=1)



Endpoint 3: logLD50

In [ ]:
%%time
a,b,c,d,e = prepare_input(train_labels, train_ecfp6_bits, target = 'logLD50_mmolkg', encoder = None)

ld50_rf_ecfp6bits = model_selection(rf_reg, rf_grid_parameters, a, c, scoring = 'neg_mean_squared_error', cv=5, GridSearch = False, n_iter=30,n_jobs=1)


In [ ]:
%%time
a,b,c,d,e = prepare_input(train_labels, train_maccs, target = 'logLD50_mmolkg', encoder = None)

ld50_rf_maccs = model_selection(rf_reg, rf_grid_parameters, a, c, scoring = 'neg_mean_squared_error', cv=5, GridSearch = False, n_iter=30,n_jobs=1)


In [ ]:
%%time
a,b,c,d,e = prepare_input(train_labels, train_ecfp6_counts, target = 'logLD50_mmolkg', encoder = None)

ld50_rf_ecfp6count = model_selection(rf_reg, rf_grid_parameters, a, c, scoring = 'neg_mean_squared_error', cv=5, GridSearch = False, n_iter=30,n_jobs=1)


In [ ]:
%%time
a,b,c,d,e = prepare_input(train_labels, train_rdkit2d, target = 'logLD50_mmolkg', encoder = None)

ld50_rf_rdkit2d = model_selection(rf_reg, rf_grid_parameters, a, c, scoring = 'neg_mean_squared_error', cv=5, GridSearch = False, n_iter=30,n_jobs=1)


In [ ]:
%%time
a,b,c,d,e = prepare_input(train_labels, train_mordred, target = 'logLD50_mmolkg', encoder = None)

ld50_rf_mordred = model_selection(rf_reg, rf_grid_parameters, a, c, scoring = 'neg_mean_squared_error', cv=5, GridSearch = False, n_iter=30,n_jobs=1)



XGBoost
Endpoint 1: logLD50

In [ ]:
%%time
a,b,c,d,e = prepare_input(train_labels, train_ecfp6_bits, target = 'logLD50_mmolkg', encoder = None)

ld50_xgb_ecfp6bits = model_selection(xgb_reg, xgb_grid_parameters, a, c, scoring = 'neg_mean_squared_error', cv=5, GridSearch = False, n_iter=30, n_jobs=1)


In [ ]:
%%time
a,b,c,d,e = prepare_input(train_labels, train_maccs, target = 'logLD50_mmolkg', encoder = None)

ld50_xgb_maccs = model_selection(xgb_reg, xgb_grid_parameters, a, c, scoring = 'neg_mean_squared_error', cv=5, GridSearch = False, n_iter=30, n_jobs=1)


In [ ]:
%%time
a,b,c,d,e = prepare_input(train_labels, train_ecfp6_counts, target = 'logLD50_mmolkg', encoder = None)

ld50_xgb_ecfp6count = model_selection(xgb_reg, xgb_grid_parameters, a, c, scoring = 'neg_mean_squared_error', cv=5, GridSearch = False, n_iter=30, n_jobs=1)


In [ ]:
%%time
a,b,c,d,e = prepare_input(train_labels, train_rdkit2d, target = 'logLD50_mmolkg', encoder = None)

ld50_xgb_rdkit2d = model_selection(xgb_reg, xgb_grid_parameters, a, c, scoring = 'neg_mean_squared_error', cv=5, GridSearch = False, n_iter=30, n_jobs=1)


In [ ]:
%%time
a,b,c,d,e = prepare_input(train_labels, train_mordred, target = 'logLD50_mmolkg', encoder = None)

ld50_xgb_mordred = model_selection(xgb_reg, xgb_grid_parameters, a, c, scoring = 'neg_mean_squared_error', cv=5, GridSearch = False, n_iter=30, n_jobs=1)


Endpoint 2: toxic

In [ ]:
encoder_toxic = joblib.load('../encoder_models/encoder_toxic.joblib')

In [ ]:
%%time
a,b,c,d,e = prepare_input(train_labels, train_ecfp6_bits, target = 'toxic', encoder = encoder_toxic)

t_xgb_ecfp6bits = model_selection(xgb_clf, xgb_grid_parameters, a, c, scoring = 'roc_auc', cv=5, GridSearch = False, n_iter=30,n_jobs=1)


In [ ]:
%%time
a,b,c,d,e = prepare_input(train_labels, train_maccs, target = 'toxic', encoder = encoder_toxic)

t_xgb_maccs = model_selection(xgb_clf, xgb_grid_parameters, a, c, scoring = 'roc_auc', cv=5, GridSearch = False, n_iter=30,n_jobs=1)


In [ ]:
%%time
a,b,c,d,e = prepare_input(train_labels, train_rdkit2d, target = 'toxic', encoder = encoder_toxic)

t_xgb_rdkit2d = model_selection(xgb_clf, xgb_grid_parameters, a, c, scoring = 'roc_auc', cv=5, GridSearch = False, n_iter=30,n_jobs=1)


In [ ]:
%%time
a,b,c,d,e = prepare_input(train_labels, train_mordred, target = 'toxic', encoder = encoder_toxic)

t_xgb_mordred = model_selection(xgb_clf, xgb_grid_parameters, a, c, scoring = 'roc_auc', cv=5, GridSearch = False, n_iter=30,n_jobs=1)


In [ ]:
%%time
a,b,c,d,e = prepare_input(train_labels, train_ecfp6_counts, target = 'toxic', encoder = encoder_toxic)

t_xgb_ecfp6count = model_selection(xgb_clf, xgb_grid_parameters, a, c, scoring = 'roc_auc', cv=5, GridSearch = False, n_iter=30,n_jobs=1)


Endpoint 3: EPA

In [ ]:
encoder_epa = joblib.load('../encoder_models/encoder_epa.joblib')


In [ ]:
%%time
a,b,c,d,e = prepare_input(train_labels, train_ecfp6_bits, target = 'EPA_category', encoder = encoder_epa)

epa_xgb_ecfp6bits = model_selection(xgb_clf, xgb_grid_parameters, a, c, scoring = 'f1_weighted', cv=5, GridSearch = False, n_iter=30,n_jobs=1)


In [ ]:
%%time
a,b,c,d,e = prepare_input(train_labels, train_maccs, target = 'EPA_category', encoder = encoder_epa)

epa_xgb_maccs = model_selection(xgb_clf, xgb_grid_parameters, a, c, scoring = 'f1_weighted', cv=5, GridSearch = False, n_iter=30,n_jobs=1)


In [ ]:
%%time
a,b,c,d,e = prepare_input(train_labels, train_ecfp6_counts, target = 'EPA_category', encoder = encoder_epa)

epa_xgb_ecfp6count = model_selection(xgb_clf, xgb_grid_parameters, a, c, scoring = 'f1_weighted', cv=5, GridSearch = False, n_iter=30,n_jobs=1)


In [ ]:
%%time
a,b,c,d,e = prepare_input(train_labels, train_rdkit2d, target = 'EPA_category', encoder = encoder_epa)

epa_xgb_rdkit2d = model_selection(xgb_clf, xgb_grid_parameters, a, c, scoring = 'f1_weighted', cv=5, GridSearch = False, n_iter=30,n_jobs=1)


In [ ]:
%%time
a,b,c,d,e = prepare_input(train_labels, train_mordred, target = 'EPA_category', encoder = encoder_epa)

epa_xgb_mordred = model_selection(xgb_clf, xgb_grid_parameters, a, c, scoring = 'f1_weighted', cv=5, GridSearch = False, n_iter=30,n_jobs=1)
